## STEP 2 

In [ ]:
import os
import glob
import math
import pickle
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from IPython.display import display

In [ ]:
# ------------------------------------------------------------
# 1. Load the correctly oriented Cora graph from Step 0
# ------------------------------------------------------------

def load_directed_graph(edges_path, labels_path=None):
    """
    Load the standardized directed edge list produced by Step 0.

    Expected format:
        source, target, weight

    where:
        source = citing paper
        target = cited paper

    Therefore every stored edge already has the correct direction:
        citing -> cited

    Reciprocal citations remain distinct directed edges.
    Self-loops and exact directed duplicates are removed as a safety check.
    """

    edges_df = pd.read_csv(
        edges_path,
        sep=",",
        header=None,
        names=["source", "target", "weight"]
    )

    edges_df["source"] = edges_df["source"].astype(int)
    edges_df["target"] = edges_df["target"].astype(int)

    edges_df = (
        edges_df[
            edges_df["source"] != edges_df["target"]
        ]
        .drop_duplicates(subset=["source", "target"])
        .reset_index(drop=True)
    )

    G = nx.DiGraph()

    if labels_path is not None:
        labels_df = pd.read_csv(
            labels_path,
            sep=",",
            header=None,
            names=["node", "label"]
        )
        labels_df["node"] = labels_df["node"].astype(int)
        G.add_nodes_from(
            sorted(labels_df["node"].unique())
        )

    G.add_edges_from(
        edges_df[["source", "target"]]
        .itertuples(index=False, name=None)
    )

    print("Loaded standardized Step 0 edge list.")
    print("Citation orientation: citing -> cited")

    return G


In [ ]:
# ------------------------------------------------------------
# 2. Load directed synthetic graphs produced by the R script
# ------------------------------------------------------------

def build_position_to_node_map(graph_folder, node_ids):
    map_path = os.path.join(graph_folder, "node_id_map.csv")
    node_ids = list(map(int, node_ids))

    if os.path.exists(map_path):
        node_map = pd.read_csv(map_path)

        required = {"node_index", "node_id"}
        if not required.issubset(node_map.columns):
            raise ValueError(
                "node_id_map.csv must contain node_index and node_id."
            )

        node_map["node_index"] = node_map["node_index"].astype(int)
        node_map["node_id"] = node_map["node_id"].astype(int)

        mapping = dict(
            zip(node_map["node_index"], node_map["node_id"])
        )

        if set(mapping.values()) != set(node_ids):
            raise ValueError(
                "node_id_map.csv and the ARD matrix contain different nodes."
            )

        return mapping

    warnings.warn(
        "node_id_map.csv was not found; the ARD row order will be used."
    )

    return {
        position: node_id
        for position, node_id in enumerate(node_ids, start=1)
    }


def load_directed_synthetic_graphs_from_csv_gz(
    graph_folder,
    prefix,
    node_ids
):
    pattern = os.path.join(
        graph_folder,
        f"{prefix}_edges_*.csv.gz"
    )
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(
            f"No graph files matched: {pattern}"
        )

    node_ids = list(map(int, node_ids))
    n_nodes = len(node_ids)

    position_to_node = build_position_to_node_map(
        graph_folder,
        node_ids
    )

    graphs = []

    for file_path in tqdm(
        files,
        desc="Loading directed synthetic graphs"
    ):
        edges_df = pd.read_csv(file_path)

        G_pos = nx.DiGraph()
        G_pos.add_nodes_from(range(1, n_nodes + 1))

        if not edges_df.empty:
            if {"from", "to"}.issubset(edges_df.columns):
                source_col, target_col = "from", "to"
            else:
                source_col, target_col = edges_df.columns[:2]

            edges_df[source_col] = edges_df[source_col].astype(int)
            edges_df[target_col] = edges_df[target_col].astype(int)

            edges_df = edges_df[
                edges_df[source_col] != edges_df[target_col]
            ].drop_duplicates(
                subset=[source_col, target_col]
            )

            valid_positions = set(position_to_node.keys())
            used_positions = set(edges_df[source_col]).union(
                edges_df[target_col]
            )

            invalid = used_positions - valid_positions
            if invalid:
                raise ValueError(
                    f"Invalid node positions in {file_path}: "
                    f"{sorted(invalid)[:10]}"
                )

            G_pos.add_edges_from(
                edges_df[[source_col, target_col]]
                .itertuples(index=False, name=None)
            )

        G = nx.relabel_nodes(
            G_pos,
            position_to_node,
            copy=True
        )
        graphs.append(G)

    return graphs

In [ ]:
# ------------------------------------------------------------
# 3. Directed graph-level metrics
# ------------------------------------------------------------

def safe_metric(function, *args, **kwargs):
    try:
        return function(*args, **kwargs)
    except Exception:
        return np.nan


def largest_weak_component(G):
    if G.number_of_nodes() == 0:
        return G.copy()

    nodes = max(
        nx.weakly_connected_components(G),
        key=len
    )
    return G.subgraph(nodes).copy()


def largest_strong_component(G):
    if G.number_of_nodes() == 0:
        return G.copy()

    nodes = max(
        nx.strongly_connected_components(G),
        key=len
    )
    return G.subgraph(nodes).copy()


def reciprocal_pairs(G):
    return sum(
        1
        for u, v in G.edges()
        if G.has_edge(v, u)
    ) // 2


def directed_graph_metrics(G):
    if not G.is_directed():
        raise TypeError("A directed graph is required.")

    nodes = list(G.nodes())
    in_degree = np.array(
        [G.in_degree(node) for node in nodes],
        dtype=float
    )
    out_degree = np.array(
        [G.out_degree(node) for node in nodes],
        dtype=float
    )

    weak_components = list(
        nx.weakly_connected_components(G)
    )
    strong_components = list(
        nx.strongly_connected_components(G)
    )

    G_weak = largest_weak_component(G)
    G_strong = largest_strong_component(G)

    if G_strong.number_of_nodes() <= 1:
        directed_path = 0.0
        directed_diameter = 0
    else:
        directed_path = safe_metric(
            nx.average_shortest_path_length,
            G_strong
        )
        directed_diameter = safe_metric(
            nx.diameter,
            G_strong
        )

    G_weak_undirected = G_weak.to_undirected()

    if G_weak_undirected.number_of_nodes() <= 1:
        weak_path = 0.0
    else:
        weak_path = safe_metric(
            nx.average_shortest_path_length,
            G_weak_undirected
        )

    in_degree_centrality = nx.in_degree_centrality(G)
    out_degree_centrality = nx.out_degree_centrality(G)
    betweenness = nx.betweenness_centrality(G)

    # NetworkX uses incoming distances for directed closeness.
    incoming_closeness = nx.closeness_centrality(G)
    outgoing_closeness = nx.closeness_centrality(
        G.reverse(copy=False)
    )

    try:
        eig_in = nx.eigenvector_centrality(
            G,
            max_iter=2000
        )
        avg_eig_in = np.mean(list(eig_in.values()))
    except Exception:
        avg_eig_in = np.nan

    try:
        eig_out = nx.eigenvector_centrality(
            G.reverse(copy=False),
            max_iter=2000
        )
        avg_eig_out = np.mean(list(eig_out.values()))
    except Exception:
        avg_eig_out = np.nan

    pagerank = nx.pagerank(G)

    reciprocity_value = (
        nx.reciprocity(G)
        if G.number_of_edges() > 0
        else np.nan
    )

    return {
        "Number of Nodes": G.number_of_nodes(),
        "Number of Directed Edges": G.number_of_edges(),
        "Number of Weak Components": len(weak_components),
        "Largest Weak Component Size": G_weak.number_of_nodes(),
        "Number of Strong Components": len(strong_components),
        "Largest Strong Component Size": G_strong.number_of_nodes(),
        "Average In-Degree": in_degree.mean(),
        "Average Out-Degree": out_degree.mean(),
        "Maximum In-Degree": in_degree.max(),
        "Maximum Out-Degree": out_degree.max(),
        "Directed Density": nx.density(G),
        "Reciprocity": reciprocity_value,
        "Reciprocal Pairs": reciprocal_pairs(G),
        "Average Directed Clustering": nx.average_clustering(G),
        "Average Directed Path Length (Largest SCC)": directed_path,
        "Directed Diameter (Largest SCC)": directed_diameter,
        "Average Path Length (Largest WCC, Direction Ignored)": weak_path,
        "Average In-Degree Centrality": np.mean(
            list(in_degree_centrality.values())
        ),
        "Average Out-Degree Centrality": np.mean(
            list(out_degree_centrality.values())
        ),
        "Average Betweenness Centrality": np.mean(
            list(betweenness.values())
        ),
        "Average Incoming Closeness": np.mean(
            list(incoming_closeness.values())
        ),
        "Average Outgoing Closeness": np.mean(
            list(outgoing_closeness.values())
        ),
        "Average Incoming Eigenvector Centrality": avg_eig_in,
        "Average Outgoing Eigenvector Centrality": avg_eig_out,
        "Average PageRank": np.mean(list(pagerank.values())),
        "Out-In Degree Assortativity": safe_metric(
            nx.degree_assortativity_coefficient,
            G,
            x="out",
            y="in"
        )
    }


def compute_directed_graph_level_metrics(graphs):
    rows = []

    for graph_id, G in enumerate(
        tqdm(graphs, desc="Computing graph-level metrics"),
        start=1
    ):
        row = directed_graph_metrics(G)
        row["graph_id"] = graph_id
        rows.append(row)

    df = pd.DataFrame(rows)
    return df[
        ["graph_id"] +
        [column for column in df.columns if column != "graph_id"]
    ]

In [ ]:
def plot_graph_metric_distributions(
    metric_df,
    original_metrics
):
    metric_names = [
        column
        for column in metric_df.columns
        if column != "graph_id"
    ]

    n_cols = 3
    n_rows = math.ceil(len(metric_names) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(20, 4.5 * n_rows)
    )
    axes = np.atleast_1d(axes).ravel()

    for axis, metric_name in zip(axes, metric_names):
        values = pd.to_numeric(
            metric_df[metric_name],
            errors="coerce"
        ).dropna()

        if len(values) > 0:
            sns.histplot(
                values,
                bins=30,
                ax=axis,
                color="cornflowerblue",
                edgecolor="black",
                kde=True
            )

        original_value = original_metrics.get(
            metric_name,
            np.nan
        )

        if pd.notna(original_value):
            axis.axvline(
                original_value,
                color="red",
                linestyle="--",
                label="Original directed graph"
            )
            axis.legend()

        axis.set_title(metric_name)
        axis.set_xlabel("Value")
        axis.set_ylabel("Frequency")

    for axis in axes[len(metric_names):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------
# 4. Directed node-level metrics
# ------------------------------------------------------------

def compute_directed_node_level_metrics(G):
    metrics = {
        "in_degree": pd.Series(dict(G.in_degree()), dtype=float),
        "out_degree": pd.Series(dict(G.out_degree()), dtype=float),
        "total_degree": pd.Series(dict(G.degree()), dtype=float),
        "in_degree_centrality": pd.Series(
            nx.in_degree_centrality(G),
            dtype=float
        ),
        "out_degree_centrality": pd.Series(
            nx.out_degree_centrality(G),
            dtype=float
        ),
        "incoming_closeness": pd.Series(
            nx.closeness_centrality(G),
            dtype=float
        ),
        "outgoing_closeness": pd.Series(
            nx.closeness_centrality(G.reverse(copy=False)),
            dtype=float
        ),
        "betweenness": pd.Series(
            nx.betweenness_centrality(G),
            dtype=float
        ),
        "directed_clustering": pd.Series(
            nx.clustering(G),
            dtype=float
        ),
        "pagerank": pd.Series(
            nx.pagerank(G),
            dtype=float
        )
    }

    try:
        metrics["incoming_eigenvector"] = pd.Series(
            nx.eigenvector_centrality(G, max_iter=2000),
            dtype=float
        )
    except Exception:
        metrics["incoming_eigenvector"] = pd.Series(
            0.0,
            index=G.nodes(),
            dtype=float
        )

    try:
        metrics["outgoing_eigenvector"] = pd.Series(
            nx.eigenvector_centrality(
                G.reverse(copy=False),
                max_iter=2000
            ),
            dtype=float
        )
    except Exception:
        metrics["outgoing_eigenvector"] = pd.Series(
            0.0,
            index=G.nodes(),
            dtype=float
        )

    try:
        metrics["kcore_total_degree"] = pd.Series(
            nx.core_number(G),
            dtype=float
        )
    except Exception:
        metrics["kcore_total_degree"] = pd.Series(
            0.0,
            index=G.nodes(),
            dtype=float
        )

    return metrics


def compute_estimated_node_metrics(graphs):
    estimated = None

    for G in tqdm(
        graphs,
        desc="Computing node-level metrics"
    ):
        current = compute_directed_node_level_metrics(G)

        if estimated is None:
            estimated = {
                name: []
                for name in current
            }

        for name, values in current.items():
            estimated[name].append(values)

    return estimated


def summarize_estimated_node_metrics(estimated):
    stats = {
        name: pd.concat(series_list, axis=1)
        .agg(["mean", "std"], axis=1)
        for name, series_list in estimated.items()
    }

    means = {
        name: pd.concat(series_list, axis=1)
        .fillna(0)
        .mean(axis=1)
        for name, series_list in estimated.items()
    }

    return stats, pd.DataFrame(means)

In [ ]:
def plot_node_metric_scatter(
    original_metrics,
    estimated_stats
):
    metric_names = list(original_metrics.keys())
    n_cols = 3
    n_rows = math.ceil(len(metric_names) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 5 * n_rows)
    )
    axes = np.atleast_1d(axes).ravel()

    for axis, metric_name in zip(axes, metric_names):
        original = original_metrics[metric_name].sort_index()
        stats = estimated_stats[metric_name].sort_index()

        common_nodes = original.index.intersection(
            stats.index
        )

        x = original.loc[common_nodes]
        y = stats.loc[common_nodes, "mean"]
        yerr = stats.loc[common_nodes, "std"].fillna(0)

        axis.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o",
            alpha=0.55,
            ecolor="gray",
            capsize=2
        )

        if len(x) > 0:
            min_value = min(x.min(), y.min())
            max_value = max(x.max(), y.max())
            axis.plot(
                [min_value, max_value],
                [min_value, max_value],
                "r--",
                label="y = x"
            )

        axis.set_title(
            metric_name.replace("_", " ").title()
        )
        axis.set_xlabel("Original")
        axis.set_ylabel("Estimated mean ± std")
        axis.grid(True)
        axis.legend()

    for axis in axes[len(metric_names):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


def compute_node_metric_correlations(
    original_metrics,
    estimated_means
):
    correlations = {}

    for metric_name, original in original_metrics.items():
        original = original.sort_index()
        estimated = estimated_means[metric_name].sort_index()

        common_nodes = original.index.intersection(
            estimated.index
        )

        correlations[metric_name] = (
            original.loc[common_nodes].corr(
                estimated.loc[common_nodes]
            )
        )

    return pd.Series(
        correlations,
        name="pearson_correlation"
    ).sort_values(ascending=False)


def compute_node_metric_errors(
    original_metrics,
    estimated_means
):
    errors = {}

    for metric_name, original in original_metrics.items():
        original = original.sort_index()
        estimated = estimated_means[metric_name].sort_index()

        common_nodes = original.index.intersection(
            estimated.index
        )

        original = original.loc[common_nodes]
        estimated = estimated.loc[common_nodes]

        absolute_error = (estimated - original).abs()
        percentage_error = (
            absolute_error / (original.abs() + 1e-8)
        )

        errors[metric_name] = {
            "absolute_error": absolute_error,
            "percentage_error": percentage_error
        }

    return errors

In [ ]:
# ------------------------------------------------------------
# 5. NSUM estimates of outgoing degree
# ------------------------------------------------------------

def estimate_out_degrees_nsum(
    ard_matrix,
    membership_matrix,
    method="mean_of_ratios"
):
    ard = ard_matrix.copy()
    membership = membership_matrix.copy()

    ard.index = ard.index.astype(int)
    membership.index = membership.index.astype(int)

    membership = membership.loc[
        ard.index,
        ard.columns
    ]

    p_k = membership.sum(axis=0) / membership.shape[0]

    valid_groups = p_k[p_k > 0].index
    ard = ard[valid_groups]
    p_k = p_k[valid_groups]

    if method == "ratio_of_sums":
        estimates = ard.sum(axis=1) / p_k.sum()

    elif method == "mean_of_ratios":
        estimates = ard.div(p_k, axis=1).mean(axis=1)

    else:
        raise ValueError(
            "method must be mean_of_ratios or ratio_of_sums."
        )

    estimates = (
        estimates
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(lower=0)
    )

    estimates.name = "estimated_out_degree"
    return estimates


def estimate_target_directed_edges(
    out_degree_estimates,
    n_nodes
):
    # Directed edges equal the sum of out-degrees.
    target_edges = int(round(out_degree_estimates.sum()))

    max_edges = n_nodes * (n_nodes - 1)

    return min(max(target_edges, 0), max_edges)


def integer_target_out_degrees(
    out_degree_estimates,
    n_nodes
):
    return (
        out_degree_estimates
        .round()
        .astype(int)
        .clip(lower=0, upper=n_nodes - 1)
    )

## Directed Adamic–Adar

In [ ]:
def directed_adamic_adar_scores_for_source(
    G,
    source
):
    scores = {}

    for intermediary in G.successors(source):
        intermediary_degree = (
            G.in_degree(intermediary)
            + G.out_degree(intermediary)
        )

        if intermediary_degree <= 1:
            continue

        contribution = (
            1.0 / math.log(intermediary_degree)
        )

        for target in G.successors(intermediary):
            if target == source:
                continue

            if G.has_edge(source, target):
                continue

            scores[target] = (
                scores.get(target, 0.0)
                + contribution
            )

    return scores


def add_directed_adamic_adar_edges_to_target(
    G,
    target_edges,
    target_out_degrees
):
    if not G.is_directed():
        raise TypeError("A directed graph is required.")

    new_G = G.copy()
    original_edges = new_G.number_of_edges()
    n_to_add = target_edges - original_edges

    if n_to_add <= 0:
        return new_G, {
            "original_edges": original_edges,
            "target_edges": target_edges,
            "added_edges": 0,
            "added_by_directed_adamic_adar": 0,
            "added_by_fallback": 0,
            "final_edges": original_edges
        }

    target_out_degrees = (
        target_out_degrees
        .reindex(new_G.nodes())
        .fillna(0)
        .astype(int)
    )

    remaining_deficit = {
        node: max(
            int(target_out_degrees.loc[node])
            - new_G.out_degree(node),
            0
        )
        for node in new_G.nodes()
    }

    candidates = []

    for source, deficit in remaining_deficit.items():
        if deficit <= 0:
            continue

        scores = directed_adamic_adar_scores_for_source(
            new_G,
            source
        )

        for target, score in scores.items():
            candidates.append(
                (
                    float(score),
                    new_G.in_degree(target),
                    source,
                    target
                )
            )

    candidates.sort(
        key=lambda item: (item[0], item[1]),
        reverse=True
    )

    added_total = 0
    added_by_aa = 0

    for _, _, source, target in candidates:
        if added_total >= n_to_add:
            break

        if remaining_deficit.get(source, 0) <= 0:
            continue

        if source == target or new_G.has_edge(source, target):
            continue

        new_G.add_edge(source, target)
        remaining_deficit[source] -= 1
        added_total += 1
        added_by_aa += 1

    # Fallback only when positive directed AA candidates are insufficient.
    added_by_fallback = 0

    if added_total < n_to_add:
        source_order = sorted(
            new_G.nodes(),
            key=lambda node: (
                remaining_deficit.get(node, 0),
                -new_G.out_degree(node)
            ),
            reverse=True
        )

        target_order = sorted(
            new_G.nodes(),
            key=lambda node: (
                new_G.in_degree(node),
                new_G.out_degree(node)
            ),
            reverse=True
        )

        progress = True

        while added_total < n_to_add and progress:
            progress = False

            for source in source_order:
                if added_total >= n_to_add:
                    break

                if remaining_deficit.get(source, 0) <= 0:
                    continue

                selected_target = None

                for target in target_order:
                    if source == target:
                        continue

                    if new_G.has_edge(source, target):
                        continue

                    selected_target = target
                    break

                if selected_target is None:
                    remaining_deficit[source] = 0
                    continue

                new_G.add_edge(source, selected_target)
                remaining_deficit[source] -= 1
                added_total += 1
                added_by_fallback += 1
                progress = True

    # Final safety fallback if rounded source targets were insufficient.
    if added_total < n_to_add:
        sources = sorted(
            new_G.nodes(),
            key=lambda node: new_G.out_degree(node)
        )
        targets = sorted(
            new_G.nodes(),
            key=lambda node: new_G.in_degree(node),
            reverse=True
        )

        for source in sources:
            if added_total >= n_to_add:
                break

            for target in targets:
                if added_total >= n_to_add:
                    break

                if source == target or new_G.has_edge(source, target):
                    continue

                new_G.add_edge(source, target)
                added_total += 1
                added_by_fallback += 1

    info = {
        "original_edges": original_edges,
        "target_edges": target_edges,
        "added_edges": added_total,
        "added_by_directed_adamic_adar": added_by_aa,
        "added_by_fallback": added_by_fallback,
        "final_edges": new_G.number_of_edges()
    }

    return new_G, info


def augment_directed_graphs(
    graphs,
    target_edges,
    target_out_degrees
):
    augmented_graphs = []
    rows = []

    for graph_id, G in enumerate(
        tqdm(graphs, desc="Applying directed Adamic-Adar"),
        start=1
    ):
        G_aug, info = (
            add_directed_adamic_adar_edges_to_target(
                G,
                target_edges,
                target_out_degrees
            )
        )

        augmented_graphs.append(G_aug)
        info["graph_id"] = graph_id
        rows.append(info)

    info_df = pd.DataFrame(rows)

    columns = [
        "graph_id",
        "original_edges",
        "target_edges",
        "added_edges",
        "added_by_directed_adamic_adar",
        "added_by_fallback",
        "final_edges"
    ]

    return augmented_graphs, info_df[columns]

In [ ]:
# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

def save_augmented_graphs(graphs, output_path):
    with open(output_path, "wb") as file:
        pickle.dump(graphs, file)


def save_added_edges_info(info_df, output_path):
    info_df.to_csv(output_path, index=False)

In [ ]:
# ------------------------------------------------------------
# 8. Input paths
# ------------------------------------------------------------

ard_path = "ard_matrix_cora_directed_out.csv"
membership_path = "membership_matrix_cora_directed.csv"

# IMPORTANT:
# This is the standardized directed edge list generated by Step 0.
# It already stores source=citing and target=cited.
edges_path = "cora_directed_edges.csv"
labels_path = "cora.node_labels"

graph_folder = "cora_directed_synthetic_graphs"
graph_prefix = "cora_directed"

output_folder = "cora_directed_link_prediction"
os.makedirs(output_folder, exist_ok=True)

NSUM_METHOD = "mean_of_ratios"

ard_matrix = pd.read_csv(
    ard_path,
    index_col=0
)

membership_matrix = pd.read_csv(
    membership_path,
    index_col=0
)

ard_matrix.index = ard_matrix.index.astype(int)
membership_matrix.index = membership_matrix.index.astype(int)

node_ids = ard_matrix.index.tolist()
n_nodes = ard_matrix.shape[0]

print("=== Input dimensions ===")
print("ARD:", ard_matrix.shape)
print("Membership:", membership_matrix.shape)
print("Nodes:", n_nodes)
print("NSUM method:", NSUM_METHOD)
print("Original directed edge file:", edges_path)
print("Expected orientation: citing -> cited")


In [ ]:
# ------------------------------------------------------------
# 9. Load original and synthetic directed graphs
# ------------------------------------------------------------

G_original = load_directed_graph(
    edges_path,
    labels_path
)

synthetic_graphs = (
    load_directed_synthetic_graphs_from_csv_gz(
        graph_folder,
        graph_prefix,
        node_ids
    )
)

print("Synthetic graphs:", len(synthetic_graphs))
print("Original nodes:", G_original.number_of_nodes())
print("Original directed edges:", G_original.number_of_edges())
print("Original is directed:", G_original.is_directed())
print("Original citation orientation: citing -> cited")

if synthetic_graphs:
    print(
        "Example synthetic directed edges:",
        synthetic_graphs[0].number_of_edges()
    )
    print(
        "Example is directed:",
        synthetic_graphs[0].is_directed()
    )


In [ ]:
# ------------------------------------------------------------
# 10. Safety checks
# ------------------------------------------------------------

original_nodes = set(G_original.nodes())
ard_nodes = set(node_ids)

if original_nodes != ard_nodes:
    raise ValueError(
        "The original graph and ARD matrix contain different nodes."
    )

if not G_original.is_directed():
    raise TypeError(
        "The original Cora graph must be directed."
    )

if nx.number_of_selfloops(G_original) > 0:
    raise ValueError(
        "The original directed graph contains self-loops."
    )

# ------------------------------------------------------------
# Critical orientation check
# ------------------------------------------------------------
# The seven Cora groups are mutually exclusive and exhaustive.
# Therefore, for every node i:
#
#   sum_k ARD_out[i,k] = out_degree(i)
#
# If the citation direction were reversed anywhere between Step 0
# and Step 2, this check would fail.
ard_out_counts = ard_matrix.sum(axis=1).astype(int)

original_out_degrees = pd.Series(
    dict(G_original.out_degree()),
    dtype=int
).reindex(ard_matrix.index)

orientation_mismatch = (
    ard_out_counts != original_out_degrees
)

if orientation_mismatch.any():
    mismatch_nodes = orientation_mismatch[
        orientation_mismatch
    ].index.tolist()

    example_nodes = mismatch_nodes[:10]

    comparison = pd.DataFrame({
        "ard_row_sum": ard_out_counts.loc[example_nodes],
        "original_out_degree": original_out_degrees.loc[example_nodes]
    })

    print(comparison)

    raise ValueError(
        "Outgoing ARD row sums do not match the out-degrees "
        "of the original graph. Check citation orientation."
    )

if int(ard_out_counts.sum()) != G_original.number_of_edges():
    raise ValueError(
        "The total number of outgoing ARD counts does not match "
        "the number of directed edges in the original graph."
    )

for graph_id, G in enumerate(synthetic_graphs, start=1):
    if not G.is_directed():
        raise TypeError(
            f"Synthetic graph {graph_id} is not directed."
        )

    if set(G.nodes()) != original_nodes:
        raise ValueError(
            f"Synthetic graph {graph_id} has different nodes."
        )

    if nx.number_of_selfloops(G) > 0:
        raise ValueError(
            f"Synthetic graph {graph_id} contains self-loops."
        )

print("All direction and node-alignment checks passed.")
print(
    "Outgoing ARD validated: each ARD row sum equals "
    "the true out-degree of the corresponding citing paper."
)


In [ ]:
# ------------------------------------------------------------
# 11. Diagnostics before correction
# ------------------------------------------------------------

diagnostics_before = os.path.join(
    output_folder,
    "diagnostics_before_directed_adamic_adar"
)
os.makedirs(diagnostics_before, exist_ok=True)

graph_metrics_before = (
    compute_directed_graph_level_metrics(
        synthetic_graphs
    )
)

true_graph_metrics = directed_graph_metrics(
    G_original
)

graph_metrics_before.to_csv(
    os.path.join(
        diagnostics_before,
        "synthetic_directed_graph_level_metrics.csv"
    ),
    index=False
)

pd.Series(
    true_graph_metrics,
    name="value"
).to_csv(
    os.path.join(
        diagnostics_before,
        "original_directed_graph_level_metrics.csv"
    )
)

print("=== Original directed graph metrics ===")
display(pd.Series(true_graph_metrics, name="value"))

plot_graph_metric_distributions(
    graph_metrics_before,
    true_graph_metrics
)

In [ ]:
original_node_metrics = (
    compute_directed_node_level_metrics(
        G_original
    )
)

estimated_node_metrics_before = (
    compute_estimated_node_metrics(
        synthetic_graphs
    )
)

with open(
    os.path.join(
        diagnostics_before,
        "synthetic_directed_node_metrics.pkl"
    ),
    "wb"
) as file:
    pickle.dump(estimated_node_metrics_before, file)

(
    estimated_stats_before,
    estimated_means_before
) = summarize_estimated_node_metrics(
    estimated_node_metrics_before
)

estimated_means_before.to_csv(
    os.path.join(
        diagnostics_before,
        "synthetic_directed_node_metric_means.csv"
    )
)

plot_node_metric_scatter(
    original_node_metrics,
    estimated_stats_before
)

correlations_before = (
    compute_node_metric_correlations(
        original_node_metrics,
        estimated_means_before
    )
)

print("=== Correlations before correction ===")
display(correlations_before)

correlations_before.to_csv(
    os.path.join(
        diagnostics_before,
        "directed_node_metric_correlations.csv"
    )
)

errors_before = compute_node_metric_errors(
    original_node_metrics,
    estimated_means_before
)

In [ ]:
# ------------------------------------------------------------
# 12. NSUM + directed Adamic-Adar
# ------------------------------------------------------------

out_degree_estimates = estimate_out_degrees_nsum(
    ard_matrix,
    membership_matrix,
    method=NSUM_METHOD
)

target_out_degrees = integer_target_out_degrees(
    out_degree_estimates,
    n_nodes
)

target_edges = estimate_target_directed_edges(
    out_degree_estimates,
    n_nodes
)

nsum_summary = pd.DataFrame({
    "node": out_degree_estimates.index,
    "estimated_out_degree": out_degree_estimates.values,
    "integer_target_out_degree": target_out_degrees.values,
    "original_out_degree": [
        G_original.out_degree(node)
        for node in out_degree_estimates.index
    ]
})

nsum_summary.to_csv(
    os.path.join(
        output_folder,
        "nsum_out_degree_estimates.csv"
    ),
    index=False
)

print(
    "Estimated target directed edges:",
    target_edges
)
print(
    "Original directed graph edges:",
    G_original.number_of_edges()
)
print(
    "Sum of rounded target out-degrees:",
    int(target_out_degrees.sum())
)

display(nsum_summary.head())

augmented_graphs, added_edges_df = (
    augment_directed_graphs(
        synthetic_graphs,
        target_edges,
        target_out_degrees
    )
)

display(added_edges_df.describe())

In [ ]:
# ------------------------------------------------------------
# 13. Save augmented directed graphs
# ------------------------------------------------------------

graphs_output_path = os.path.join(
    output_folder,
    "graphs_directed_adamic_adar_nsum.pkl"
)

info_output_path = os.path.join(
    output_folder,
    "directed_adamic_adar_added_edges_info.csv"
)

save_augmented_graphs(
    augmented_graphs,
    graphs_output_path
)

save_added_edges_info(
    added_edges_df,
    info_output_path
)

print("Saved:", graphs_output_path)
print("Saved:", info_output_path)

In [ ]:
# ------------------------------------------------------------
# 14. Diagnostics after correction
# ------------------------------------------------------------

diagnostics_after = os.path.join(
    output_folder,
    "diagnostics_after_directed_adamic_adar"
)
os.makedirs(diagnostics_after, exist_ok=True)

for graph_id, G in enumerate(augmented_graphs, start=1):
    if not G.is_directed():
        raise TypeError(
            f"Augmented graph {graph_id} is not directed."
        )

    if set(G.nodes()) != original_nodes:
        raise ValueError(
            f"Augmented graph {graph_id} has different nodes."
        )

    if nx.number_of_selfloops(G) > 0:
        raise ValueError(
            f"Augmented graph {graph_id} contains self-loops."
        )

graph_metrics_after = (
    compute_directed_graph_level_metrics(
        augmented_graphs
    )
)

graph_metrics_after.to_csv(
    os.path.join(
        diagnostics_after,
        "augmented_directed_graph_level_metrics.csv"
    ),
    index=False
)

pd.Series(
    true_graph_metrics,
    name="value"
).to_csv(
    os.path.join(
        diagnostics_after,
        "original_directed_graph_level_metrics.csv"
    )
)

plot_graph_metric_distributions(
    graph_metrics_after,
    true_graph_metrics
)

In [ ]:
estimated_node_metrics_after = (
    compute_estimated_node_metrics(
        augmented_graphs
    )
)

with open(
    os.path.join(
        diagnostics_after,
        "augmented_directed_node_metrics.pkl"
    ),
    "wb"
) as file:
    pickle.dump(estimated_node_metrics_after, file)

(
    estimated_stats_after,
    estimated_means_after
) = summarize_estimated_node_metrics(
    estimated_node_metrics_after
)

estimated_means_after.to_csv(
    os.path.join(
        diagnostics_after,
        "augmented_directed_node_metric_means.csv"
    )
)

plot_node_metric_scatter(
    original_node_metrics,
    estimated_stats_after
)

correlations_after = (
    compute_node_metric_correlations(
        original_node_metrics,
        estimated_means_after
    )
)

print("=== Correlations after correction ===")
display(correlations_after)

correlations_after.to_csv(
    os.path.join(
        diagnostics_after,
        "directed_node_metric_correlations_after.csv"
    )
)

errors_after = compute_node_metric_errors(
    original_node_metrics,
    estimated_means_after
)

print("Step 2 completed.")